In [35]:
import sys
import pandas as pd
from IPython.display import display

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

pd.set_option("display.unicode.east_asian_width", True)

input_file = r"c:\Users\HP\Desktop\data haraka\Rslt_Mvt_Enseignant_Prim2025_cleaned.xlsx"

print("Loading data...")
df = pd.read_excel(input_file, dtype=str)

df.columns = [col.strip() for col in df.columns]

# Convert numeric columns
if "Points" in df.columns:
    df["Points"] = pd.to_numeric(df["Points"], errors="coerce")
if "Choice_Rank" in df.columns:
    df["Choice_Rank"] = pd.to_numeric(df["Choice_Rank"], errors="coerce")

# Helpers to safely find columns
def _normalize(name):
    return "".join(str(name).lower().split())

def _find_col_contains(columns, keywords):
    for col in columns:
        norm = _normalize(col)
        if all(k in norm for k in keywords):
            return col
    return None

col_assignment_commune = _find_col_contains(df.columns, ["assignment", "commune"]) or _find_col_contains(
    df.columns, ["commune"]
)
col_full_name = _find_col_contains(df.columns, ["full", "name"]) or _find_col_contains(
    df.columns, ["name"]
)
col_points = _find_col_contains(df.columns, ["points"])
col_choice_rank = _find_col_contains(df.columns, ["choice", "rank"]) or _find_col_contains(
    df.columns, ["rank"]
)
col_original_province = _find_col_contains(df.columns, ["original", "province"]) or _find_col_contains(
    df.columns, ["province"]
)
col_request_type = _find_col_contains(df.columns, ["request", "type"]) or _find_col_contains(
    df.columns, ["request_type"]
)

missing_cols = [
    name
    for name, col in {
        "Assignment Commune": col_assignment_commune,
        "Full_Name": col_full_name,
        "Points": col_points,
        "Choice_Rank": col_choice_rank,
        "Original_Province": col_original_province,
        "Request_Type": col_request_type,
    }.items()
    if col is None
]

if missing_cols:
    print("Missing columns:", missing_cols)
    print("Available columns:")
    display(pd.Series(df.columns))
else:
    # Table for Assignment Commune عين عتيق (البلدية)
    commune_name = ")المقاطعة( أكدال الرياν"
    commune_mask = df[col_assignment_commune].astype(str).str.contains(
        commune_name, na=False, regex=False
    )
    commune_table = df.loc[
        commune_mask,
        [col_full_name, col_points, col_choice_rank, col_request_type, col_assignment_commune, col_original_province],
    ].drop_duplicates().sort_values([col_choice_rank, col_points], ascending=[True, False])

    print(f"Rows for {commune_name}: {len(commune_table)}")
    display(commune_table)

    print("\nOriginal_Province values for this commune:")
    display(commune_table[col_original_province].dropna().unique())


Loading data...
Rows for )المقاطعة( أكدال الرياν: 1


,Full_Name,Points,Choice_Rank,Request_Type,Assignment Commune,Original_Province
3057,حسنا˯ السعداوي,96,1,اϹلتحاق بالزوج,)المقاطعة( أكدال الرياν,المنυر الجميل )رائدة (



Original_Province values for this commune:


<StringArray>
['المنυر الجميل  )رائدة (']
Length: 1, dtype: str